In [0]:
import json
from pyspark.sql import Row
from pyspark.sql.functions import lag, sum as spark_sum, min as spark_min, max as spark_max, lit as spark_lit, col, when, row_number, current_timestamp
from pyspark.sql.window import Window

# Dim Player History (traded or waived and signed with another team)

In [0]:
batting = spark.table("silver.mlb_batting_stats")
pitching = spark.table("silver.mlb_pitching_stats")

# Combining batting & pitching appearances (team_id per player per game)
games_lookup_for_scd = spark.table("silver.mlb_schedule").select("game_pk", "game_date")

real_team_ids = [row.team_id for row in spark.table("silver.mlb_teams").select("team_id").collect()]

player_team_events = (
    batting.select("player_id", "player_name", "team_id", "game_pk")
    .union(pitching.select("player_id", "player_name", "team_id", "game_pk"))
    .dropDuplicates(["player_id", "game_pk"])   # a two-way player only needs one event per game
    .join(games_lookup_for_scd, on="game_pk", how="left")
    # Dropping All-Star Game / exhibition appearances that use special "team" ids and are not real MLB clubs.
    .filter(col("team_id").isin(real_team_ids))
)

In [0]:
player_window = Window.partitionBy("player_id").orderBy("game_date")

events_with_change_flag = (
    player_team_events
    .withColumn("prev_team_id", lag("team_id").over(player_window))
    .withColumn(
        "is_new_stint",
        when(col("prev_team_id").isNull(), spark_lit(1))  # first game with that team
        .when(col("team_id") != col("prev_team_id"), spark_lit(1))  # Changed Teams
        .otherwise(spark_lit(0))
    )
    .withColumn("stint_group", spark_sum("is_new_stint").over(player_window))
)

In [0]:
stints = (
    events_with_change_flag
    .groupBy("player_id", "player_name", "team_id", "stint_group")
    .agg(
        spark_min("game_date").alias("valid_from"),
        spark_max("game_date").alias("valid_to")
    )
)

In [0]:
# Identfying each player's most recent stint (highest stint_group number)
latest_stint_window = Window.partitionBy("player_id").orderBy(col("stint_group").desc())

dim_players_history = (
    stints
    .withColumn("row_num", row_number().over(latest_stint_window))
    .withColumn(
        "valid_to",
        when(col("row_num") == 1, spark_lit(None).cast("string")).otherwise(col("valid_to"))
    )
    .withColumn("is_current", col("row_num") == 1)
    .drop("row_num", "stint_group")
    .withColumn("gold_processed_at", current_timestamp())
)

In [0]:
dim_players_history.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.dim_players_history")

In [0]:
print(f"dim_players_history rows: {dim_players_history.count()}")
display(dim_players_history.orderBy("player_id", "valid_from").limit(50))

# Validation: Finding players with more than one stint (team)

In [0]:
all_stat_team_ids = (
    batting.select("team_id").union(pitching.select("team_id")).distinct()
)

dropped_team_ids = all_stat_team_ids.filter(~col("team_id").isin(real_team_ids))

print(f"Non-club team_ids found in stats (filtered out of player history): {dropped_team_ids.count()}")
display(dropped_team_ids)

In [0]:
traded_players = (
    dim_players_history
    .groupBy("player_id", "player_name")
    .count()
    .filter(col("count") > 1)
)

print(f"Players with multiple stints (likely trades): {traded_players.count()}")
display(traded_players)

In [0]:
%sql
SELECT h.player_name, h.team_id, t.team_name, h.valid_from, h.valid_to, h.is_current
FROM gold.dim_players_history h
JOIN gold.dim_teams t ON h.team_id = t.team_id
WHERE t.team_name = 'Los Angeles Dodgers'
   AND '2026-08-10' BETWEEN h.valid_from AND COALESCE(h.valid_to, '2099-12-31')
ORDER BY h.player_name